# SatQuery — full official BigEarthNet v2 download
Run on your education account, using a CPU runtime. Choose the same education account when mounting Drive.
Downloads all optical, SAR, reference maps and both metadata tables (about 118 GB) into MyDrive/SatQuery/bigearthnet-v2-full-official. No satellite data is downloaded to your Mac.
Files are saved as ordered 256 MiB parts with SHA256 receipts; the complete byte sequence is verified against each official MD5. Parts are not individually extractable archives. Later processing must read them in order. No tensors or features are generated here.
If Colab disconnects, rerun this cell: verified parts are reused. A Drive quota/write error stops the run. Free runtime duration is not guaranteed. Keep Drive desktop syncing off for this folder if you do not want local copies.
Source: https://zenodo.org/records/10891137


In [ ]:
from pathlib import Path
import hashlib, json, time, os
import requests
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/SatQuery/bigearthnet-v2-full-official')
ROOT.mkdir(parents=True, exist_ok=True)
PART = 256 * 1024 * 1024
FILES = [{'name': 'BigEarthNet-S1.tar.zst', 'size': 54439153171, 'md5': 'a55eaa2cdf6a917e296bd6601ec1e348', 'url': 'https://zenodo.org/api/records/10891137/files/BigEarthNet-S1.tar.zst/content'}, {'name': 'metadata.parquet', 'size': 3616349, 'md5': '55687065e77b6d0b0f1ff604a6e7b49c', 'url': 'https://zenodo.org/api/records/10891137/files/metadata.parquet/content'}, {'name': 'metadata_for_patches_with_snow_cloud_or_shadow.parquet', 'size': 710162, 'md5': 'fe31856f4986d446c9468b59d6387c91', 'url': 'https://zenodo.org/api/records/10891137/files/metadata_for_patches_with_snow_cloud_or_shadow.parquet/content'}, {'name': 'Reference_Maps.tar.zst', 'size': 282391301, 'md5': '95d85a222fa983faddcac51a19f28917', 'url': 'https://zenodo.org/api/records/10891137/files/Reference_Maps.tar.zst/content'}, {'name': 'BigEarthNet-S2.tar.zst', 'size': 63251710377, 'md5': '2245ed2d1a93f6ce637d839bc856396e', 'url': 'https://zenodo.org/api/records/10891137/files/BigEarthNet-S2.tar.zst/content'}]
(ROOT / 'source-manifest.json').write_text(json.dumps(FILES, indent=2))
print(f'Total official download: {sum(f["size"] for f in FILES)/1e9:.3f} GB', flush=True)
for item in FILES:
    folder = ROOT / (item['name'] + '.parts')
    folder.mkdir(exist_ok=True)
    whole = hashlib.md5()
    for index, start in enumerate(range(0, item['size'], PART)):
        end = min(start + PART, item['size']) - 1
        expected = end - start + 1
        path = folder / f'{index:05d}.part'
        receipt = folder / f'{index:05d}.sha256'
        valid = False
        if path.exists() and receipt.exists() and path.stat().st_size == expected:
            digest = hashlib.sha256()
            with path.open('rb') as stream:
                for block in iter(lambda: stream.read(8*1024*1024), b''):
                    digest.update(block)
            valid = digest.hexdigest() == receipt.read_text().strip()
        if not valid:
            temp = folder / f'{index:05d}.partial'
            for attempt in range(6):
                try:
                    with requests.get(item['url'], headers={'Range': f'bytes={start}-{end}', 'Accept-Encoding':'identity'}, stream=True, timeout=(30,120)) as response:
                        response.raise_for_status()
                        if response.status_code != 206 or response.headers.get('Content-Range') != f'bytes {start}-{end}/{item["size"]}':
                            raise RuntimeError('Server did not honor the requested byte range; stopped safely.')
                        digest = hashlib.sha256()
                        count = 0
                        with temp.open('wb') as output:
                            for block in response.iter_content(4*1024*1024):
                                count += len(block)
                                if count > expected:
                                    raise RuntimeError('Server sent more bytes than requested.')
                                output.write(block)
                                digest.update(block)
                            output.flush()
                            os.fsync(output.fileno())
                        if count != expected:
                            raise requests.ConnectionError('Incomplete range response')
                    temp.replace(path)
                    receipt.write_text(digest.hexdigest())
                    break
                except requests.RequestException:
                    if attempt == 5:
                        raise
                    print('Network retry; completed parts remain saved.', flush=True)
                    time.sleep(min(2**attempt,30))
            # Storage/write errors deliberately stop instead of repeatedly writing.
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(8*1024*1024), b''):
                whole.update(block)
        print(f'{item["name"]}: {(end+1)/1e9:.3f}/{item["size"]/1e9:.3f} GB verified', flush=True)
    if whole.hexdigest() != item['md5']:
        raise RuntimeError(f'Official checksum mismatch: {item["name"]}. Do not use these parts.')
    (folder / 'VERIFIED.json').write_text(json.dumps(item, indent=2))
    print('Official checksum verified:', item['name'], flush=True)
print('All five official files downloaded and verified. Archives remain in ordered parts; no extraction performed.', flush=True)
